# Machine Failure Cost Lab

Review the recorded benchmark without refitting or retuning. Synthetic current-failure labels; no advance-warning claim. See the README and method card for limitations.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "failurelab").exists():
    ROOT = ROOT.parent
assert (ROOT / "failurelab").exists(), "Start from the repository or notebooks directory"
sys.path.insert(0, str(ROOT))
results = json.loads((ROOT / "reports/results.json").read_text())
print("Selected model:", results["selected_model"])
print("Frozen threshold:", results["threshold"])


Selected model: boosting_physics
Frozen threshold: 0.011716755082956706


In [2]:
rows = [{"model": name, "selection_AP": results["selection"][name]["average_precision"],
         "test_AP": metrics["average_precision"], "test_AUC": metrics["roc_auc"]}
        for name, metrics in results["test_models"].items()]
print(pd.DataFrame(rows).to_string(index=False))
p = results["test_policy"]
print(f"Detected: {p['tp']} | missed: {p['fn']} | false alerts: {p['fp']}")
print(f"Recall: {p['recall']:.1%} | precision: {p['precision']:.1%}")


           model  selection_AP  test_AP  test_AUC
      prevalence      0.020667 0.019500  0.500000
        logistic      0.515794 0.412920  0.882804
    boosting_raw      0.422503 0.590278  0.960290
boosting_physics      0.610953 0.635225  0.954033
Detected: 34 | missed: 5 | false alerts: 118
Recall: 87.2% | precision: 22.4%


## Policy tradeoff

![Cost and error counts](../reports/alert_policy.svg)

The 25:1 cost ratio is illustrative. The threshold was selected on 1,000 separate rows containing only 22 failures.

In [3]:
sensitivity = [{"miss_cost": row["miss_cost"], "threshold": row["threshold"],
                "test_cost": row["test"]["cost"], "recall": row["test"]["recall"],
                "false_alerts": row["test"]["fp"]}
               for row in results["cost_sensitivity"]]
print(pd.DataFrame(sensitivity).to_string(index=False))
print("Conditional 95% intervals:", results["uncertainty"]["intervals_95"])


 miss_cost  threshold  test_cost   recall  false_alerts
         5   0.354038       92.0 0.615385            17
        10   0.354038      167.0 0.615385            17
        25   0.011717      243.0 0.871795           118
        50   0.011717      368.0 0.871795           118
       100   0.011717      618.0 0.871795           118
Conditional 95% intervals: {'average_precision': [0.42125555422361927, 0.7982953614595194], 'recall': [0.7670348837209303, 0.9583333333333334], 'cost_per_1000': [77.45, 169.2625]}


## Score the sensor example

The example contains the first ten test-block rows. It is a replay of benchmark inputs, not new validation data. The saved JSON artifact does not execute pickle code.

In [4]:
from failurelab.models import Model
bundle = json.loads((ROOT / "reports/model.json").read_text())
sensors = pd.read_csv(ROOT / "examples/sensors.csv")
scores = Model.from_dict(bundle["model"]).predict_proba(sensors)[:, 1]
output = pd.DataFrame({"UDI": sensors["UDI"], "risk_score": scores,
                       "alert": (scores >= bundle["threshold"]).astype(int)})
print(output.to_string(index=False))


 UDI  risk_score  alert
8001    0.001736      0
8002    0.001771      0
8003    0.000634      0
8004    0.267389      1
8005    0.002032      0
8006    0.000719      0
8007    0.001255      0
8008    0.001616      0
8009    0.005798      0
8010    0.101988      1


## Reproduce separately

Run from the repository root:

    python -m unittest discover -s tests -v
    python -m failurelab train --output reports/reproduction

The current test block is already disclosed. Further model development needs a new untouched evaluation dataset.